In [1]:
import sonnet as snt
import tensorflow as tf
tf.compat.v1.enable_eager_execution()
import numpy as np

In [2]:
from migration.datasets import create_AIS_dataset
inputs, targets, _, _, _, lengths, mean =  create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                   '../../data/ct_2017010203_10_20/mean.pkl',
                   32,
                   99999, # not used lol
                   300,
                   300, 
                   30,
                   72, 
                   shuffle=False,
                   repeat=False)


Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf.py_func
    (it is not differentiable, and manipulates numpy arrays). It drops the
    stateful argument making all functions stateful.
    
Instructions for updating:
Use `tf.cast` instead.
Instructions for updating:
Use `tf.cast` instead.
Instructions for updating:
Use `for ... in dataset:` to iterate over a dataset. If using `tf.estimator`, return the `Dataset` object directly from your input function. As a last resort, you can use `tf.compat.v1.data.make_one_

2025-07-02 12:39:41.464616: I tensorflow/core/platform/cpu_feature_guard.cc:142] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 FMA
2025-07-02 12:39:41.468816: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 2688010000 Hz
2025-07-02 12:39:41.470404: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x18b24380 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-07-02 12:39:41.470431: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version


In [4]:
seq_lengths = lengths
batch_size = tf.shape(input=seq_lengths)[0]
max_seq_len = tf.reduce_max(input_tensor=seq_lengths)

# shape (t, B) of 1 and 0
seq_mask = tf.transpose(
        a=tf.sequence_mask(seq_lengths, maxlen=max_seq_len, dtype=tf.float32),
        perm=[1, 0])

In [5]:
latent_size = 64
_DEFAULT_INITIALIZERS = {"w": tf.compat.v1.keras.initializers.VarianceScaling(scale=1.0, mode="fan_avg", distribution="uniform",seed=111),
                         "b": tf.compat.v1.zeros_initializer()}

In [6]:
rnn_cell = tf.compat.v1.nn.rnn_cell.LSTMCell(latent_size, initializer=_DEFAULT_INITIALIZERS["w"])

Instructions for updating:
This class is equivalent as tf.keras.layers.LSTMCell, and will be replaced by that in Tensorflow 2.0.


In [22]:
# The zero state of the model: LSTM state, output of the latent_feat_extractor perceptron
rnn_state = rnn_cell.zero_state(32, dtype=tf.float32)
prev_latent_encoded = tf.zeros((32, 64), dtype=tf.float32)


inputs_encoded = tf.random.uniform((32,64), seed=1)
targets_encoded = tf.random.uniform((32,64), seed=1)
rnn_inputs = tf.concat([inputs_encoded, prev_latent_encoded], axis=1)

In [24]:
rnn_inputs

<tf.Tensor: id=268, shape=(32, 128), dtype=float32, numpy=
array([[0.6914257 , 0.825331  , 0.955783  , ..., 0.        , 0.        ,
        0.        ],
       [0.10626292, 0.94774735, 0.80372286, ..., 0.        , 0.        ,
        0.        ],
       [0.9710717 , 0.8872111 , 0.3762871 , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.24392152, 0.96074533, 0.7361698 , ..., 0.        , 0.        ,
        0.        ],
       [0.23129177, 0.03118658, 0.45489275, ..., 0.        , 0.        ,
        0.        ],
       [0.08502483, 0.11556482, 0.49520338, ..., 0.        , 0.        ,
        0.        ]], dtype=float32)>

In [25]:
rnn_state

LSTMStateTuple(c=<tf.Tensor: id=145, shape=(32, 64), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>, h=<tf.Tensor: id=151, shape=(32, 64), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>)

In [26]:
rnn_out, new_rnn_state = rnn_cell(rnn_inputs, rnn_state)

In [34]:
rnn_out

<tf.Tensor: id=291, shape=(32, 64), dtype=float32, numpy=
array([[ 0.0229526 , -0.12647128,  0.02183165, ..., -0.10462476,
         0.06549463, -0.01467913],
       [ 0.02476343, -0.0915398 ,  0.05322585, ..., -0.06169284,
         0.14303422,  0.01969768],
       [-0.00482575, -0.1120412 ,  0.0338759 , ..., -0.09342836,
         0.11844736, -0.06228068],
       ...,
       [ 0.00227056, -0.0325354 ,  0.0133897 , ..., -0.07863463,
         0.10833255, -0.11219411],
       [ 0.03727942, -0.05175337,  0.01773514, ..., -0.03818294,
         0.10530294, -0.04346327],
       [ 0.02542332, -0.0518475 ,  0.0591781 , ..., -0.11231359,
         0.07109517, -0.0469728 ]], dtype=float32)>

In [28]:
rnn_state

LSTMStateTuple(c=<tf.Tensor: id=145, shape=(32, 64), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>, h=<tf.Tensor: id=151, shape=(32, 64), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>)